This notebook provides alignment, conversion, and analysis of videos using the video2vision pipeline. It is intended to walk you through all the steps of using the system. The first cell prompts you to set several important parameters:

 * **Experiment Name:** This is a unique identifier used to cache experimental parameters in case the processing is interrupted before it completes. If you set this value, do some processing, close the notebook, and then come back later, if you reenter the experiment name, the other parameters you set should immediately be restored. However, this must be unique.
 * **Alignment Pipeline:** The path to the JSON file containing the pipeline used to align the UV to the visible. Two pipelines are provided by default in the data folder, `still_alignment_pipeline.json` for still images and `video_alignment_pipeline.json` for videos. New alignment pipelines can be created if needed using the Alignment-Pipeline-Builder notebook.
 * **Visible Input Path:** The path to the visible images or video. This can be a path to a single file, a path to a folder (in which case all files in that folder will be processed), or it can be a path including a wildcard, such as `/path/to/images/*.JPEG`.
 * **UV Input Path:** The path to the UV images or video. If the visible and UV input paths point to multiple files, then those files must match each other when ordered lexicographically. For example, if visible input path points to a folder containing `01.JPEG` and `02.JPEG`, and UC input path points to a folder containing `A.JPEG` and `B.JPEG`, then `01.JPEG` must match `A.JPEG` and `02.JPEG` must match `B.JPEG`.
 * **Aligned UV Output Path:** The folder to write the aligned UV outputs to. The new files will be given names by merging the names of the matched visible and UV input files. For example, in our previous example, the output files would be named `01-A.JPEG` and `02-B.JPEG`.

In [1]:
%gui tk

import matplotlib.pyplot as plot
import numpy as np

import v2v_nb

vis_linearization_selector = uv_linearization_selector = vis_test_selector = uv_test_selector = line_op = None

config = v2v_nb.get_config()
# v2v_nb.ConfigBox('experiment_name', 'align_pipe_path', 'vis_path', 'uv_path', 'uv_aligned_path', show_buttons=True)

config['experiment_name'] = '20261103'
config['align_pipe_path'] = '/Users/syc/Desktop/GoogleSync/NMNH/equipment/setup_test/20261103/aruco/alignment.json'
config['vis_path'] = '/Users/syc/Desktop/GoogleSync/NMNH/equipment/setup_test/20261103/aruco/VIS/*.ARW'
config['uv_path'] = '/Users/syc/Desktop/GoogleSync/NMNH/equipment/setup_test/20261103/aruco/UV/*.ARW'
config['uv_aligned_path'] = '/Users/syc/Desktop/GoogleSync/NMNH/equipment/setup_test/20261103/aruco/UV_aligned/'

We begin by running the alignment. This will take a while if it hasn't been previously run. The alignment operator will be cached when the alignment is finished running, and a report of how long each operator took to run will be printed. The **Alignment Warp** and **Time Shift** entries should not be filled in when the notebook is first loaded; they will be filled in automatically by the alignment process.

In [2]:
v2v_nb.ConfigBox('coe', 'shift')

ConfigBox(children=(ArrayBox(children=(Label(value='Alignment Warp'), HBox(children=(ArrayText(children=(Text(…

In [ ]:
v2v_nb.build_and_run_alignment_pipeline()

With the alignment run, let's visualize some of the images to make sure everything looks good. The left image is the image from the visible camera, the right image is the image from the ultraviolet camera, and the middle image is the image from the ultraviolet camera aligned to the visible image. The buttons below the display allow moving forward and backward in time in a video.

In [3]:
display(v2v_nb.make_initial_displaybox())

DisplayBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00#L\x00\x00\x07\xdc\x08\x02\x00\x…

It can be hard to tell from a side-by-side display how accurately two images are aligned. This next cell displays a composite image formed by using the red and blue channels from the visible image, and the green channel from the aligned ultraviolet image. If the alignment is inaccurate, there should be visible "ghosting" caused by objects not perfectly overlaying each other.

In [4]:
display(v2v_nb.make_ghostbox())

GhostBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x0b\xc4\x00\x00\x07\xdc\x08\x02\x…

Now we need to select the samples for linearization. This requires setting some additional paths:

 * **Visible Linearization Path:** The path to the visible inputs to be used in calculating the linearization. This can be the same as Visible Input Path.
 * **UV Linearization Path:** The path to the UV inputs to be used in calculating the linearization.
 * **Linearization Sample Autolocator Path:** The path to a JSON containing an autolinearizer that will use fiducial markers, such as ARUCO tags, to automatically locate the sample points. The autolinearizer for our default ARUCO board is found in the data folder, as `autolinearizer.json`. Autolinearizers can be created for new ARUCO boards using the Autolinearizer-Builder notebook. If an autolinearizer is not available, you can manually select sample points, as described below. In that case, just leave this entry blank.
 * **Linearization Sample Values Path:** The path to a csv file containing the expected reflectances of each sample used for linearization. The values for our current ARUCO board can be found in the data folder as `aruco_samples.csv`.
 * **Camera Sensitivities Path:** The path to a csv file containing the per-wavelength sensitivities of the camera. The values for our current Sony camera can be found in the data folder as `camera_sensitivities.csv`.
 * **Use Sony SLog3 Linearization:** This specifies whether the camera uses the Sony SLog3 function for gamma compression. This is determined by the camera hardware. If true, we can use this function to linearize the camera pixels without needing to fit coefficients, which is generally more accurate. We still need the samples for normalization, however. If false, we use the linearization sample values path to fit a polynomial to estimate the linearization function.
 * **Do Not Linearize:** If checked, the input files are assumed to be in raw format, and to not need linearization. They will still be normalized.

In [5]:
# v2v_nb.ConfigBox(
#     'vis_linearization_path', 'uv_linearization_path', 'linearization_auto_op_path', 'linearization_values_path',
#     'camera_path', 'is_sony_camera', 'normalize_only',
# )

config['vis_linearization_path'] = '/Users/syc/Desktop/GoogleSync/NMNH/equipment/setup_test/20261103/aruco/VIS/VS_02269.ARW'
config['uv_linearization_path'] = '/Users/syc/Desktop/GoogleSync/NMNH/equipment/setup_test/20261103/aruco/UV/UV_02267.ARW'
config['linearization_auto_op_path'] = None
config['linearization_values_path'] = '/Users/syc/Desktop/GoogleSync/Projects/2026_Bird_color_gap/video2vision/data/fluorilon_standards.csv'
config['camera_path'] = '/Users/syc/Desktop/GoogleSync/Projects/2026_Bird_color_gap/video2vision/data/camera_sensitivities.csv'
config['is_sony_camera'] = False
config['normalize_only'] = True

In [6]:
vis_linearization_selector = v2v_nb.make_selectorbox('vis_linearization_path')
display(vis_linearization_selector)

SelectorBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x0b\xc4\x00\x00\x07\xdc\x08\x0…

The selections will be automatically transferred to the ultraviolet image using the alignment operation. Check to make sure they look right, and correct if necessary before continuing.

In [7]:
uv_linearization_selector = v2v_nb.make_selectorbox(
    'uv_linearization_path', copy_from=vis_linearization_selector
)
display(uv_linearization_selector)

SelectorBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x0b\xc4\x00\x00\x07\xdc\x08\x0…

We next extract the pixel intensities from each sample point and build the linearization.

In [8]:
line_op = v2v_nb.build_linearizer(vis_linearization_selector, uv_linearization_selector)

Could not find /Users/syc/Desktop/GoogleSync/Projects/2026_Bird_color_gap/video2vision/data/fluorilon_fws99_corrected.csv - please check path.


We now compare the linearized values to the expected values.

In [ ]:
linearization_expected_values, linearized_sample_values, linearization_samples, table = v2v_nb.evaluate_samples(
    line_op, config['linearization_values_path'],
    vis_linearization_selector, uv_linearization_selector,
)

plot.figure(figsize=(18, 6))
plot.subplot(1, 3, 3)
for band, band_name in enumerate(['UV', 'Blue', 'Green', 'Red']):
    color = (band_name[0].lower() if band_name != 'UV' else 'purple')
    plot.scatter(
        linearization_expected_values[:, band],
        linearized_sample_values[:, band],
        label=band_name,
        color=color,
        s=100,
    )

plot.xlabel('Actual Value')
plot.ylabel('Linearized Value')

plot.subplot(1, 3, 2)
for band, band_name in enumerate(['UV', 'Blue', 'Green', 'Red']):
    color = (band_name[0].lower() if band_name != 'UV' else 'purple')
    plot.scatter(
        linearization_samples[:, band],
        linearized_sample_values[:, band],
        label=band_name,
        color=color,
        s=100,
    )

plot.xlabel('Measured Value')
plot.ylabel('Linearized Value')

plot.subplot(1, 3, 1)
for band, band_name in enumerate(['UV', 'Blue', 'Green', 'Red']):
    color = (band_name[0].lower() if band_name != 'UV' else 'purple')
    plot.scatter(
        linearization_samples[:, band],
        linearization_expected_values[:, band],
        label=band_name,
        color=color,
        s=100,
    )

plot.xlabel('Measured Value')
plot.ylabel('Actual Value')

plot.show()

print(table)

Let's run an image through the linearizer to see what it looks like.

In [ ]:
display(v2v_nb.make_example_linearization_images(line_op))

We now test the linearizer on another color sample. Once again, select the samples, taking care to keep in order. These next few cells can be skipped if you do not wish to test the linearization error. If you do, set the following values first:

 * **Visible Test Path:** The path to the visible inputs to be used in evaluating the linearization. This can be the same as Visible Input Path and the Visible Linearization Path, although they should use different samples if possible.
 * **UV Test Path:** The path to the UV inputs to be used in evaluating the linearization.
 * **Test Sample Autolocator Path:** The path to a JSON containing an autolinearizer that will use fiducial markers, such as ARUCO tags, to automatically locate the sample points.
 * **Test Sample Values Path:** The path to a csv file containing the expected reflectances of each sample used for evaluating. The values for our spectralons can be found in the data folder as `spectralon_samples.csv`.

In [ ]:
# v2v_nb.ConfigBox('vis_test_path', 'uv_test_path', 'test_auto_op_path', 'test_values_path')

config['vis_test_path'] = '/Users/syc/Desktop/GoogleSync/NMNH/equipment/setup_test/20261103/aruco/VIS/VS_02304.ARW'
config['uv_test_path'] = '/Users/syc/Desktop/GoogleSync/NMNH/equipment/setup_test/20261103/aruco/UV/UV_02302.ARW'
config['test_auto_op_path'] = '/Users/syc/Desktop/GoogleSync/NMNH/equipment/setup_test/20261103/aruco/autolinearizer_color.json'
config['test_values_path'] = '/Users/syc/Desktop/GoogleSync/Projects/2026_Bird_color_gap/video2vision/data/ARUCO_Brazil_colors_hanleylab.csv'

In [ ]:
vis_test_selector = v2v_nb.make_selectorbox('vis_test_path')
display(vis_test_selector)

In [ ]:
uv_test_selector = v2v_nb.make_selectorbox(
    'uv_test_path', copy_from=vis_test_selector
)
display(uv_test_selector)

In [ ]:
test_samples_expected_values, test_samples_linearized_values, test_samples, table = v2v_nb.evaluate_samples(
    line_op, config['test_values_path'],
    vis_test_selector, uv_test_selector,
)

plot.figure(figsize=(18, 6))
plot.subplot(1, 3, 3)
for band, band_name in enumerate(['UV', 'Blue', 'Green', 'Red']):
    color = (band_name[0].lower() if band_name != 'UV' else 'purple')
    plot.scatter(
        test_samples_expected_values[:, band],
        test_samples_linearized_values[:, band],
        label=band_name,
        color=color,
        s=100,
    )

plot.xlabel('Actual Value')
plot.ylabel('Linearized Value')

plot.subplot(1, 3, 2)
for band, band_name in enumerate(['UV', 'Blue', 'Green', 'Red']):
    color = (band_name[0].lower() if band_name != 'UV' else 'purple')
    plot.scatter(
        test_samples[:, band], #[idxs, band],
        test_samples_linearized_values[:, band],
        label=band_name,
        color=color,
        s=100,
    )

plot.xlabel('Measured Value')
plot.ylabel('Linearized Value')

plot.subplot(1, 3, 1)
for band, band_name in enumerate(['UV', 'Blue', 'Green', 'Red']):
    color = (band_name[0].lower() if band_name != 'UV' else 'purple')
    plot.scatter(
        test_samples[:, band], #[idxs, band],
        test_samples_expected_values[:, band],
        label=band_name,
        color=color,
        s=100,
    )

plot.xlabel('Measured Value')
plot.ylabel('Actual Value')

plot.show()

print(table)

Next, we construct the operator to convert linearized camera values to animal values. First, set:

 * **Animal Sensitivities Path:** The path to a csv containing the per-wavelength sensitivities of the color receptors in the animal's eyes. Suitable csvs can be found in the animal_sensitivities subfolder of the data folder.
 * **Sense Converter Path:** The path to a JSON containing an automatic sense converter. Suitable converters can be found in the converters subfolder of the data folder. New converters can be built using the Sense-Converter-Builder notebook.

In [ ]:
# v2v_nb.ConfigBox('animal_sensitivity_path', 'sense_converter_path')
config['animal_sensitivity_path'] = '/Users/syc/Desktop/GoogleSync/Projects/2026_Bird_color_gap/video2vision/data/animal_sensitivities/bluetit_sensitivities.csv'
config['sense_converter_path'] = '/Users/syc/Desktop/GoogleSync/Projects/2026_Bird_color_gap/video2vision/data/converters/bluetit_converter.json'

We visualize the sensitivity of the camera and the animal's photoreceptors.

In [ ]:
plot.figure(figsize=(15, 6))

if config['camera_path'] is None:
    print('Camera sensitivity path not set.')
else:
    plot.subplot(1, 2, 1)
    camera_sense = v2v_nb.utils.load_csv(config['camera_path'])
    for band, band_name in enumerate(['UV', 'Blue', 'Green', 'Red']):
        color = (band_name[0].lower() if band_name != 'UV' else 'purple')
        plot.plot(
            np.arange(300, 701),
            camera_sense[:, band],
            color=color,
            label=band_name,
        )
    plot.xlabel('Wavelength ($\mu$m)')
    plot.ylabel('Receptivity')
    plot.title('Camera')
    plot.legend()

if config['animal_sensitivity_path'] is None:
    print('Animal sensitivity path not set.')
else:
    plot.subplot(1, 2, 2)
    animal_sense = v2v_nb.utils.load_csv(config['animal_sensitivity_path'])
    for band in range(animal_sense.shape[1]):
        plot.plot(
            np.arange(300, 701),
            animal_sense[:, band],
            label=f'Band {band}',
        )
    plot.xlabel('Wavelength ($\mu$m)')
    plot.ylabel('Receptivity')
    plot.title('Animal')
    plot.legend()

plot.show()

Finally, we estimate the error on the test samples. Skip this cell if you did not provide test samples.

In [ ]:
expected_values, predicted_values, colors, table = v2v_nb.processing.evaluate_conversion(
    line_op, config['linearization_values_path'], vis_linearization_selector, uv_linearization_selector,
)

plot.figure(figsize=(18, 6))
for band in range(animal_sense.shape[1]):
    plot.subplot(1, animal_sense.shape[1], band + 1)
    plot.title(f'Band {band}')
    plot.xlabel('Actual values')
    plot.ylabel('Estimated values')
    plot.scatter(
        expected_values[:, band],
        predicted_values[:, band],
        s=100,
        c=colors,
        edgecolors=np.array([[0, 0, 0]]),
    )

plot.show()

print(table)

This next cell is optional, and just saves the sample point pixel values, their linearized values, converted values, etc. in a csv file. If you want to do that, use **Sample Record Path** to specify where.

In [ ]:
display(v2v_nb.ConfigBox('sample_record_path'))

In [ ]:
v2v_nb.create_record(
    line_op, vis_linearization_selector, uv_linearization_selector,
    vis_test_selector, uv_test_selector,
)

We now construct a pipeline that linearizes and converts the aligned images, and run it. This will take a while. Specify:

 * **Animal Vision Output Path:** The path to write the outputs converted to animal vision. If the animal has more than three colors of photoreceptor, then each color will be written out as a separate file, with a suffix denoting the band.
 * **Human Vision Output Path:** The path to write the outputs converted to human vision.

In [ ]:
display(v2v_nb.ConfigBox('animal_out_path', 'human_out_path'))

In [ ]:
v2v_nb.build_and_run_full_pipeline(line_op)

Let's take a look at the results:

In [ ]:
display(v2v_nb.processing.make_final_displaybox())